%md
# Age_Glaucoma: build the Zeiss and age-matched CLSA cohorts

This first-stage notebook reuses the completed Zeiss RETFound vectors
produced by `filter_color_fundus_v3`. The Zeiss images are **not** passed
through the CLSA quality pipeline because the acquisition formats are
fundamentally different. Successful presence in a completed Zeiss chunk
is treated as evidence that the source-specific Zeiss preprocessing and
RETFound pipeline accepted the image. The notebook then selects
visit-specific CLSA comparator records that are observed
negative for the released major ocular conditions and matches them to
Zeiss patients on age without replacement.

Important interpretation: unless a diagnosis field is supplied below,
the Zeiss arm is a *Zeiss source cohort*, not yet a verified glaucoma
cohort. Likewise, CLSA controls are *screen-negative for the released
variables*, not proven free of all ocular disease. Missing screening
responses are never interpreted as healthy.

The existing embeddings are retained exactly as generated. No Zeiss DICOM
decoding, image-quality rerun, or RETFound inference occurs here. CLSA
images continue to require their own completed CLSA quality-pass flag.


%md
## Runtime

This notebook performs no DICOM decoding or model inference, so GPU
compute is unnecessary. It only needs the standard Databricks pandas,
PyArrow, and Spark environment.


In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from pyspark.sql import functions as F


In [ ]:
dbutils.widgets.text(
    "repo_root",
    "/Workspace/Users/ad0038@pennmedicine.upenn.edu/CLSA/CLSA_retina",
)
dbutils.widgets.text(
    "zeiss_checkpoint_root",
    "/Volumes/ophthalmology_analytics/dev_optic/zeiss_export_debug/"
    "Color_Fundus",
)
dbutils.widgets.text(
    "zeiss_embedding_chunks",
    "",
    "Optional exact chunks folder; blank auto-selects the newest completed run",
)
dbutils.widgets.text(
    "output_root",
    "/Volumes/ophthalmology_analytics/dev_optic/clsa_dataset/derived/"
    "clsa_retinal_aging/Age_Glaucoma",
)
dbutils.widgets.text(
    "clsa_quality_path",
    "/Volumes/ophthalmology_analytics/dev_optic/clsa_dataset/derived/"
    "clsa_retinal_aging/fundus_retfound/01_quality/"
    "fundus_quality_manifest.parquet",
)
dbutils.widgets.text(
    "clsa_embeddings_path",
    "/Volumes/ophthalmology_analytics/dev_optic/clsa_dataset/derived/"
    "clsa_retinal_aging/fundus_retfound/02_embeddings/"
    "retfound_embeddings_delta",
    "Delta, consolidated Parquet, or 02_embeddings root; batches are auto-discovered",
)
dbutils.widgets.text(
    "sap_path",
    "/Volumes/ophthalmology_analytics/dev_optic/clsa_dataset/derived/"
    "clsa_retinal_aging/sap_questionnaire_visit",
)
dbutils.widgets.text(
    "baseline_questionnaire_path",
    "/Volumes/ophthalmology_analytics/dev_optic/clsa_dataset/derived/"
    "clsa_retinal_aging/questionnaire_extracted_sap/BL/"
    "2209017_UOttawa_EFreeman_Baseline_CoPv7_Qx_CANUE_PA_BS.csv",
)
dbutils.widgets.text(
    "fup1_questionnaire_path",
    "/Volumes/ophthalmology_analytics/dev_optic/clsa_dataset/derived/"
    "clsa_retinal_aging/questionnaire_extracted_sap/F1/"
    "2209017_UOttawa_EFreeman_FUP1_CoPv4_Qx_PA_BS.csv",
)
dbutils.widgets.text("match_ratio", "1")
dbutils.widgets.text("age_caliper_years", "1.0")
dbutils.widgets.dropdown("exact_sex_match", "false", ["false", "true"])
dbutils.widgets.text(
    "zeiss_diagnosis_column",
    "",
    "Optional verified diagnosis column in the Zeiss embedding chunks",
)
dbutils.widgets.text(
    "zeiss_case_values",
    "",
    "Optional comma-separated accepted values for the diagnosis column",
)


In [ ]:
repo_root = Path(dbutils.widgets.get("repo_root").strip())
zeiss_checkpoint_root = Path(
    dbutils.widgets.get("zeiss_checkpoint_root").strip()
)
configured_zeiss_chunks = dbutils.widgets.get("zeiss_embedding_chunks").strip()
if configured_zeiss_chunks:
    zeiss_embedding_chunks = Path(configured_zeiss_chunks)
else:
    chunk_candidates = sorted(
        zeiss_checkpoint_root.glob("automorph_rerun_*/chunks"),
        reverse=True,
    )
    chunk_candidates.append(zeiss_checkpoint_root / "chunks")
    zeiss_embedding_chunks = next(
        (
            candidate
            for candidate in chunk_candidates
            if candidate.exists() and next(candidate.glob("chunk_*.parquet"), None)
        ),
        None,
    )
    if zeiss_embedding_chunks is None:
        raise FileNotFoundError(
            "No completed Zeiss chunk directory was found under "
            f"{zeiss_checkpoint_root}. Set zeiss_embedding_chunks explicitly."
        )
print("Zeiss embedding chunks:", zeiss_embedding_chunks)
output_root = Path(dbutils.widgets.get("output_root").strip())
clsa_quality_path = dbutils.widgets.get("clsa_quality_path").strip()
clsa_embeddings_path = dbutils.widgets.get("clsa_embeddings_path").strip()
sap_path = dbutils.widgets.get("sap_path").strip()
baseline_questionnaire_path = dbutils.widgets.get(
    "baseline_questionnaire_path"
).strip()
fup1_questionnaire_path = dbutils.widgets.get(
    "fup1_questionnaire_path"
).strip()
match_ratio = int(dbutils.widgets.get("match_ratio"))
age_caliper_years = float(dbutils.widgets.get("age_caliper_years"))
exact_sex_match = dbutils.widgets.get("exact_sex_match") == "true"
zeiss_diagnosis_column = dbutils.widgets.get("zeiss_diagnosis_column").strip()
zeiss_case_values = {
    value.strip().upper()
    for value in dbutils.widgets.get("zeiss_case_values").split(",")
    if value.strip()
}

if match_ratio < 1:
    raise ValueError("match_ratio must be at least 1")
if age_caliper_years < 0:
    raise ValueError("age_caliper_years cannot be negative")
output_root.mkdir(parents=True, exist_ok=True)

module_root = repo_root / "src"
if not module_root.exists():
    raise FileNotFoundError(f"Repository source directory not found: {module_root}")
if str(module_root) not in sys.path:
    sys.path.insert(0, str(module_root))

from age_glaucoma_cohort import (  # noqa: E402
    greedy_age_match,
    load_zeiss_embedding_chunks,
)
from fundus_retfound_pipeline import write_frame  # noqa: E402


%md
## 1. Read the completed Zeiss RETFound vectors


In [ ]:
zeiss_embeddings = load_zeiss_embedding_chunks(
    zeiss_embedding_chunks,
    expected_embedding_dim=1024,
)
print(
    f"Zeiss embeddings: {len(zeiss_embeddings):,} images, "
    f"{zeiss_embeddings['patient_id'].nunique():,} patients"
)

if zeiss_diagnosis_column:
    if zeiss_diagnosis_column not in zeiss_embeddings.columns:
        raise ValueError(
            f"Configured Zeiss diagnosis column is absent: {zeiss_diagnosis_column}"
        )
    if not zeiss_case_values:
        raise ValueError(
            "zeiss_case_values must be supplied with zeiss_diagnosis_column"
        )
    diagnosis = (
        zeiss_embeddings[zeiss_diagnosis_column].astype(str).str.strip().str.upper()
    )
    zeiss_embeddings = zeiss_embeddings[diagnosis.isin(zeiss_case_values)].copy()
    print(
        f"After verified diagnosis filter: {len(zeiss_embeddings):,} images, "
        f"{zeiss_embeddings['patient_id'].nunique():,} patients"
    )
else:
    print(
        "No Zeiss diagnosis column configured. This notebook will label these "
        "records as the Zeiss source cohort, not as confirmed glaucoma cases."
    )


%md
## 2. Accept the completed Zeiss source-pipeline embeddings

Every row here already passed far enough through the attached Zeiss
workflow to receive a valid 1,024-element RETFound vector and be written
to a completed `chunk_*.parquet`. Source quality fields such as
`automorph_retina_fraction` and `quality_score`, when present, are retained
for audit but are not reinterpreted using CLSA thresholds.


In [ ]:
zeiss_passed_images = zeiss_embeddings.copy()
zeiss_passed_images["zeiss_source_pipeline_pass"] = True
zeiss_passed_images["zeiss_source_pipeline_basis"] = (
    "valid_1024_element_embedding_in_completed_chunk"
)
write_frame(
    zeiss_passed_images,
    output_root / "01_zeiss_source_cohort" / "zeiss_embedded_images.parquet",
)
print(
    f"Zeiss source-pipeline embedded images: {len(zeiss_passed_images):,}; "
    f"patients: {zeiss_passed_images['patient_id'].nunique():,}"
)


In [ ]:
def first_observed(series):
    observed = series.dropna()
    return observed.iloc[0] if len(observed) else None


zeiss_passed_images["age"] = pd.to_numeric(
    zeiss_passed_images["age"], errors="coerce"
)
patient_aggregations = {
    "age": ["median", "min", "max"],
    "dcm_path": "count",
}
for optional_column in ("sex", "race"):
    if optional_column in zeiss_passed_images.columns:
        patient_aggregations[optional_column] = first_observed

zeiss_patients = (
    zeiss_passed_images.groupby("patient_id", as_index=False)
    .agg(patient_aggregations)
)
zeiss_patients.columns = [
    "patient_id",
    "age",
    "age_min",
    "age_max",
    "n_quality_pass_images",
    *(["sex"] if "sex" in zeiss_passed_images.columns else []),
    *(["race"] if "race" in zeiss_passed_images.columns else []),
]
zeiss_patients["age_range_years"] = (
    zeiss_patients["age_max"] - zeiss_patients["age_min"]
)
zeiss_patients["source_cohort_label"] = (
    "verified_diagnosis_filter"
    if zeiss_diagnosis_column
    else "zeiss_source_cohort_diagnosis_pending"
)
zeiss_patients = zeiss_patients.dropna(subset=["age"]).reset_index(drop=True)
write_frame(
    zeiss_patients,
    output_root / "01_zeiss_source_cohort" / "zeiss_patient_cohort.parquet",
)
display(zeiss_patients.describe(include="all").transpose())


%md
## 3. Derive conservative visit-matched CLSA ocular controls

Required observed-negative fields are:

- BL: retinal detachment, cataract history/current cataract, glaucoma,
  and macular degeneration.
- F1: the same fields plus diabetic retinopathy.
- Both visits: `visual_impairment_self_report == 0` in the SAP table.

Released codes `1` and `11` are treated as positive, and `2` as negative.
Text `yes`/`no` is also recognized. Refused, unknown, missing, and other
sentinel values remain missing, so incomplete screens are ineligible.
IOP is retained for later sensitivity analyses but is not converted into
an unprespecified disease threshold here.


In [ ]:
baseline_raw = (
    spark.read.option("header", True)
    .option("inferSchema", False)
    .csv(baseline_questionnaire_path)
)
fup1_raw = (
    spark.read.option("header", True)
    .option("inferSchema", False)
    .csv(fup1_questionnaire_path)
)
sap = spark.read.format("delta").load(sap_path)


def released_binary(df, names):
    existing = [name for name in names if name in df.columns]
    if not existing:
        return F.lit(None).cast("int")
    values = [F.upper(F.trim(F.col(name).cast("string"))) for name in existing]
    positive = None
    negative = None
    for value in values:
        is_positive = value.isin("1", "1.0", "11", "11.0", "YES", "Y", "TRUE")
        is_negative = value.isin("2", "2.0", "NO", "N", "FALSE")
        positive = is_positive if positive is None else (positive | is_positive)
        negative = is_negative if negative is None else (negative | is_negative)
    return F.when(positive, F.lit(1)).when(negative, F.lit(0)).otherwise(
        F.lit(None).cast("int")
    )


def combine_binary(*columns):
    positive = None
    negative = None
    for column in columns:
        is_positive = column == 1
        is_negative = column == 0
        positive = is_positive if positive is None else (positive | is_positive)
        negative = is_negative if negative is None else (negative | is_negative)
    return F.when(positive, F.lit(1)).when(negative, F.lit(0)).otherwise(
        F.lit(None).cast("int")
    )


bl_cataract = combine_binary(
    released_binary(baseline_raw, ["ICQ_CATRCT_COM"]),
    released_binary(baseline_raw, ["ICQ_CATRCT2_COM"]),
)
baseline_screen = baseline_raw.select(
    F.trim(F.col("entity_id").cast("string")).alias("participant_id"),
    F.lit("BL").alias("visit"),
    released_binary(baseline_raw, ["ICQ_DERET3MO_COM"]).alias(
        "retinal_detachment"
    ),
    bl_cataract.alias("cataract"),
    released_binary(baseline_raw, ["ICQ_GLAUC_COM"]).alias("glaucoma"),
    released_binary(baseline_raw, ["CCC_MACDEG_COM"]).alias(
        "macular_degeneration"
    ),
    F.lit(None).cast("int").alias("diabetic_retinopathy"),
)

f1_cataract = combine_binary(
    released_binary(fup1_raw, ["ICQ_CATRCT_COF1"]),
    released_binary(fup1_raw, ["ICQ_CATRCT2_COF1"]),
)
f1_dr = released_binary(fup1_raw, ["DIA_DIABRT_COF1"])
if "CCC_DIAB_DRAGE_NB_COF1" in fup1_raw.columns:
    f1_dr_age = F.col("CCC_DIAB_DRAGE_NB_COF1").cast("double")
    f1_dr = F.when(f1_dr_age >= 0, F.lit(1)).otherwise(f1_dr)
followup1_screen = fup1_raw.select(
    F.trim(F.col("entity_id").cast("string")).alias("participant_id"),
    F.lit("F1").alias("visit"),
    released_binary(fup1_raw, ["ICQ_DERET3MO_COF1"]).alias(
        "retinal_detachment"
    ),
    f1_cataract.alias("cataract"),
    combine_binary(
        released_binary(fup1_raw, ["ICQ_GLAUC_COF1"]),
        released_binary(fup1_raw, ["CCC_GLAUC_COF1"]),
    ).alias("glaucoma"),
    released_binary(fup1_raw, ["CCC_MACDEG_COF1"]).alias(
        "macular_degeneration"
    ),
    f1_dr.alias("diabetic_retinopathy"),
)
ocular_screen = baseline_screen.unionByName(followup1_screen)


In [ ]:
screen_fields = [
    "retinal_detachment",
    "cataract",
    "glaucoma",
    "macular_degeneration",
    "diabetic_retinopathy",
]
required_count = F.when(F.col("visit") == "BL", F.lit(4)).otherwise(F.lit(5))
observed_count = sum(
    F.when(F.col(column).isNotNull(), F.lit(1)).otherwise(F.lit(0))
    for column in screen_fields
)
positive_count = sum(
    F.when(F.col(column) == 1, F.lit(1)).otherwise(F.lit(0))
    for column in screen_fields
)
ocular_screen = (
    ocular_screen.withColumn("screen_fields_observed", observed_count)
    .withColumn("screen_fields_required", required_count)
    .withColumn(
        "screen_complete",
        F.when(
            (F.col("visit") == "BL")
            & F.col("diabetic_retinopathy").isNull()
            & (F.col("screen_fields_observed") == 4),
            F.lit(True),
        ).otherwise(F.col("screen_fields_observed") == required_count),
    )
    .withColumn("ocular_screen_positive", positive_count > 0)
)

sap_controls = sap.select(
    F.trim(F.col("participant_id").cast("string")).alias("participant_id"),
    F.upper(F.col("visit")).alias("visit"),
    F.col("age_at_fundus_years").cast("double"),
    F.col("sex_at_birth").cast("string"),
    F.col("visual_impairment_self_report").cast("double"),
)
clsa_screen = (
    ocular_screen.join(sap_controls, ["participant_id", "visit"], "inner")
    .withColumn(
        "control_eligible",
        F.col("screen_complete")
        & ~F.col("ocular_screen_positive")
        & (F.col("visual_impairment_self_report") == 0)
        & F.col("age_at_fundus_years").isNotNull(),
    )
    .withColumn(
        "control_exclusion_reason",
        F.when(~F.col("screen_complete"), F.lit("incomplete_ocular_screen"))
        .when(F.col("ocular_screen_positive"), F.lit("ocular_condition_positive"))
        .when(
            F.col("visual_impairment_self_report").isNull(),
            F.lit("visual_impairment_missing"),
        )
        .when(
            F.col("visual_impairment_self_report") != 0,
            F.lit("visual_impairment_reported"),
        )
        .when(F.col("age_at_fundus_years").isNull(), F.lit("visit_age_missing"))
        .otherwise(F.lit(None).cast("string")),
    )
)
clsa_screen.write.format("delta").mode("overwrite").partitionBy("visit").save(
    str(output_root / "03_clsa_controls" / "ocular_screen_delta")
)
display(
    clsa_screen.groupBy("visit", "control_eligible", "control_exclusion_reason")
    .agg(F.count("*").alias("records"), F.countDistinct("participant_id").alias("participants"))
    .orderBy("visit", "control_eligible", "control_exclusion_reason")
)


%md
## 4. Require completed CLSA quality and RETFound outputs


In [ ]:
def databricks_path_exists(path: str) -> bool:
    if Path(path).exists():
        return True
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False


def load_completed_clsa_quality(configured_path: str):
    """Load the consolidated CLSA QC manifest or its completed batches."""
    configured = configured_path.rstrip("/")
    if databricks_path_exists(configured):
        return spark.read.parquet(configured), configured, "consolidated_parquet"
    quality_root = (
        str(Path(configured).parent)
        if Path(configured).name == "fundus_quality_manifest.parquet"
        else configured
    )
    batch_glob = (
        f"{quality_root}/batches/batch_*/fundus_quality_manifest.parquet"
    )
    try:
        batches = spark.read.parquet(batch_glob)
        if not batches.columns:
            raise ValueError("Quality batch Parquets resolved without a schema")
    except Exception as exc:
        raise FileNotFoundError(
            "No consolidated or per-batch CLSA quality manifest was found. "
            f"Checked {configured} and {batch_glob}."
        ) from exc
    return batches, batch_glob, "completed_batch_parquets"


def load_completed_clsa_embeddings(configured_path: str):
    """Load Delta, consolidated Parquet, or completed per-batch Parquets."""
    configured = configured_path.rstrip("/")
    name = Path(configured).name
    configured_is_sample_batch = (
        name == "retfound_embeddings.parquet"
        and Path(configured).parent.name.startswith("batch_")
    )
    if configured_is_sample_batch:
        # A sample batch path represents the complete sibling batch collection.
        embeddings_root = str(Path(configured).parents[2])
        print(
            "Expanding the configured sample batch to all completed batches under:",
            embeddings_root,
        )
    elif name == "batches":
        embeddings_root = str(Path(configured).parent)
    elif name in {"retfound_embeddings_delta", "retfound_embeddings.parquet"}:
        embeddings_root = str(Path(configured).parent)
    else:
        embeddings_root = configured

    checked = []
    delta_candidates = []
    parquet_candidates = []
    if name == "retfound_embeddings_delta":
        delta_candidates.append(configured)
    elif name == "retfound_embeddings.parquet" and not configured_is_sample_batch:
        parquet_candidates.append(configured)
    elif databricks_path_exists(f"{configured}/_delta_log"):
        # A custom path may itself be a Delta directory.
        delta_candidates.append(configured)
    delta_candidates.append(f"{embeddings_root}/retfound_embeddings_delta")
    parquet_candidates.append(f"{embeddings_root}/retfound_embeddings.parquet")

    for path in dict.fromkeys(delta_candidates):
        checked.append(path)
        if databricks_path_exists(path):
            try:
                return spark.read.format("delta").load(path), path, "delta"
            except Exception as exc:
                print(
                    f"Found {path}, but it is not readable Delta: "
                    f"{type(exc).__name__}: {str(exc)[:180]}"
                )
    for path in dict.fromkeys(parquet_candidates):
        checked.append(path)
        if databricks_path_exists(path):
            return spark.read.parquet(path), path, "consolidated_parquet"

    batch_glob = (
        f"{embeddings_root}/batches/batch_*/retfound_embeddings.parquet"
    )
    checked.append(batch_glob)
    try:
        batch_embeddings_raw = spark.read.parquet(batch_glob)
        # Accessing columns forces Spark to resolve the wildcard and schema.
        if not batch_embeddings_raw.columns:
            raise ValueError("Batch Parquets resolved without a schema")
        stable_required = {"image_path", "participant_id", "visit", "embedding"}
        missing_stable = stable_required - set(batch_embeddings_raw.columns)
        if missing_stable:
            raise ValueError(
                "CLSA embedding batches lack stable required columns: "
                f"{sorted(missing_stable)}"
            )
        # Project before any action/write. Pandas/Arrow inferred fields such as
        # age as int64 in some batches and float64 in others. Reading every
        # metadata field through one Spark schema therefore raises
        # PARQUET_COLUMN_DATA_TYPE_MISMATCH. These four fields are the stable
        # matching/vector contract; age and sex are attached from the SAP table.
        selections = [
            F.col("image_path").cast("string").alias("image_path"),
            F.col("participant_id").cast("string").alias("participant_id"),
            F.col("visit").cast("string").alias("visit"),
            F.col("embedding").cast("array<float>").alias("embedding"),
        ]
        if "eye" in batch_embeddings_raw.columns:
            selections.append(F.col("eye").cast("string").alias("eye"))
        if "embedding_dim" in batch_embeddings_raw.columns:
            selections.append(
                F.col("embedding_dim").cast("int").alias("embedding_dim")
            )
        batch_embeddings = batch_embeddings_raw.select(*selections)
    except Exception as exc:
        raise FileNotFoundError(
            "No completed CLSA RETFound embedding output was found. Checked:\n- "
            + "\n- ".join(checked)
            + "\nIf the vectors were written elsewhere, set clsa_embeddings_path "
            "to that 02_embeddings directory, consolidated Parquet, or Delta path."
        ) from exc

    duplicate_paths = (
        batch_embeddings.groupBy("image_path")
        .count()
        .filter(F.col("count") > 1)
    )
    if duplicate_paths.limit(1).count():
        raise ValueError(
            "The CLSA embedding batch folders contain duplicate image paths. "
            "Remove stale batches or point clsa_embeddings_path to the correct run."
        )
    cache_path = str(output_root / "00_inputs" / "clsa_embeddings_delta")
    (
        batch_embeddings.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(cache_path)
    )
    return (
        spark.read.format("delta").load(cache_path),
        batch_glob,
        "completed_batch_parquets_cached_as_delta",
    )


clsa_quality, resolved_clsa_quality_path, clsa_quality_source_mode = (
    load_completed_clsa_quality(clsa_quality_path)
)
print("CLSA quality source mode:", clsa_quality_source_mode)
print("CLSA quality source:", resolved_clsa_quality_path)
clsa_embeddings, resolved_clsa_embeddings_path, clsa_embedding_source_mode = (
    load_completed_clsa_embeddings(clsa_embeddings_path)
)
print("CLSA embedding source mode:", clsa_embedding_source_mode)
print("CLSA embedding source:", resolved_clsa_embeddings_path)
required_embedding_columns = {"image_path", "participant_id", "visit", "embedding"}
missing_embedding_columns = required_embedding_columns - set(clsa_embeddings.columns)
if missing_embedding_columns:
    raise ValueError(
        f"CLSA embeddings are missing required columns: {sorted(missing_embedding_columns)}"
    )
if "quality_pass" not in clsa_quality.columns:
    raise ValueError("CLSA quality manifest has no quality_pass column")

quality_pass_paths = (
    clsa_quality.select("image_path", F.col("quality_pass").cast("boolean"))
    .dropDuplicates(["image_path"])
    .filter(F.col("quality_pass"))
)
clsa_eligible_images = (
    clsa_embeddings.withColumn("participant_id", F.col("participant_id").cast("string"))
    .withColumn(
        "visit",
        F.when(F.upper(F.col("visit")).isin("F1", "FUP1"), F.lit("F1"))
        .when(F.upper(F.col("visit")) == "BL", F.lit("BL"))
        .otherwise(F.lit(None).cast("string")),
    )
    .join(quality_pass_paths, "image_path", "inner")
    .join(
        clsa_screen.filter(F.col("control_eligible")),
        ["participant_id", "visit"],
        "inner",
    )
)
clsa_eligible_images.write.format("delta").mode("overwrite").partitionBy("visit").save(
    str(output_root / "03_clsa_controls" / "eligible_images_delta")
)
display(
    clsa_eligible_images.groupBy("visit").agg(
        F.count("*").alias("images"),
        F.countDistinct("participant_id").alias("participants"),
        F.avg("age_at_fundus_years").alias("mean_age"),
    )
)


%md
## 5. Match Zeiss patients to CLSA participant-visits on age

Matching is deterministic nearest-neighbor matching within the requested
caliper. A CLSA participant can be selected only once, even if that person
has eligible images at BL and F1. Sex matching is optional and off by
default because the requested first-stage match is age based.


In [ ]:
clsa_control_candidates = (
    clsa_eligible_images.groupBy("participant_id", "visit")
    .agg(
        F.first("age_at_fundus_years", ignorenulls=True).alias(
            "age_at_fundus_years"
        ),
        F.first("sex_at_birth", ignorenulls=True).alias("sex_at_birth"),
        F.count("*").alias("n_quality_pass_images"),
    )
    .toPandas()
)

match_pairs, match_audit = greedy_age_match(
    cases=zeiss_patients,
    controls=clsa_control_candidates,
    ratio=match_ratio,
    caliper_years=age_caliper_years,
    exact_sex=exact_sex_match,
)
match_root = output_root / "04_age_matching"
write_frame(match_audit, match_root / "age_match_audit.parquet")

eligible_zeiss_count = int(len(match_audit))
matched_zeiss_count = int(
    match_audit["matched"].fillna(False).astype(bool).sum()
) if eligible_zeiss_count else 0
unmatched_zeiss_count = eligible_zeiss_count - matched_zeiss_count
unmatched_percentage = (
    100.0 * unmatched_zeiss_count / eligible_zeiss_count
    if eligible_zeiss_count
    else float("nan")
)
match_run_counts = pd.DataFrame(
    [
        {
            "eligible_zeiss_patients": eligible_zeiss_count,
            "matched_zeiss_patients": matched_zeiss_count,
            "unmatched_zeiss_patients": unmatched_zeiss_count,
            "unmatched_percentage": unmatched_percentage,
            "age_caliper_years": age_caliper_years,
            "match_ratio": match_ratio,
            "exact_sex_match": exact_sex_match,
        }
    ]
)
write_frame(match_run_counts, match_root / "age_match_run_counts.csv")
print(
    f"Zeiss matching result: {matched_zeiss_count:,}/{eligible_zeiss_count:,} "
    f"matched; {unmatched_zeiss_count:,}/{eligible_zeiss_count:,} "
    f"unmatched ({unmatched_percentage:.1f}%) with a "
    f"±{age_caliper_years:g}-year caliper."
)
display(match_run_counts)
if unmatched_zeiss_count:
    unmatched_audit = match_audit[
        ~match_audit["matched"].fillna(False).astype(bool)
    ].copy()
    display(
        unmatched_audit.groupby("reason", dropna=False)
        .size()
        .rename("unmatched_zeiss_patients")
        .reset_index()
    )
    display(unmatched_audit.head(100))

if match_pairs.empty:
    raise ValueError(
        f"No age matches were found: {unmatched_zeiss_count:,} of "
        f"{eligible_zeiss_count:,} eligible Zeiss patients were unmatched "
        f"within the ±{age_caliper_years:g}-year caliper. The complete audit "
        f"was saved to {match_root / 'age_match_audit.parquet'}. Inspect the "
        "age distributions or increase age_caliper_years."
    )
write_frame(match_pairs, match_root / "age_match_pairs.parquet")
display(match_pairs.head(100))


In [ ]:
balance = pd.DataFrame(
    [
        {
            "cohort": "Zeiss matched",
            "n": int(match_pairs["zeiss_patient_id"].nunique()),
            "mean_age": float(
                match_pairs.drop_duplicates("zeiss_patient_id")[
                    "zeiss_age_years"
                ].mean()
            ),
            "sd_age": float(
                match_pairs.drop_duplicates("zeiss_patient_id")[
                    "zeiss_age_years"
                ].std()
            ),
        },
        {
            "cohort": "CLSA matched",
            "n": int(match_pairs["clsa_participant_id"].nunique()),
            "mean_age": float(match_pairs["clsa_age_years"].mean()),
            "sd_age": float(match_pairs["clsa_age_years"].std()),
        },
    ]
)
balance["mean_absolute_age_difference_years"] = float(
    match_pairs["absolute_age_difference_years"].mean()
)
balance["max_absolute_age_difference_years"] = float(
    match_pairs["absolute_age_difference_years"].max()
)
write_frame(balance, match_root / "age_match_balance.csv")
display(balance)


%md
## 6. Save image-level matched cohorts for later analyses

Outputs retain all qualifying images and their 1,024-element vectors for
each selected patient/participant-visit. The pair table is the analysis
key; later notebooks can attach verified Zeiss diagnoses, OCT layers,
longitudinal outcomes, or additional CLSA exclusions without rebuilding
the image-quality stage.


In [ ]:
match_pairs_spark = spark.createDataFrame(match_pairs)
# Read the already-written Parquet instead of inferring a Spark schema from a
# large pandas frame containing arrays and possibly all-null metadata columns.
zeiss_embeddings_spark = spark.read.parquet(
    str(
        output_root
        / "01_zeiss_source_cohort"
        / "zeiss_embedded_images.parquet"
    )
)
zeiss_match_keys = match_pairs_spark.select(
    "match_set_id",
    "zeiss_patient_id",
    "zeiss_age_years",
    "clsa_participant_id",
    "clsa_visit",
    "clsa_age_years",
    "absolute_age_difference_years",
)
zeiss_matched_images = zeiss_embeddings_spark.join(
    zeiss_match_keys,
    zeiss_embeddings_spark["patient_id"]
    == zeiss_match_keys["zeiss_patient_id"],
    "inner",
).drop(zeiss_match_keys["zeiss_patient_id"])

clsa_matched_images = clsa_eligible_images.join(
    match_pairs_spark,
    (clsa_eligible_images["participant_id"] == match_pairs_spark["clsa_participant_id"])
    & (clsa_eligible_images["visit"] == match_pairs_spark["clsa_visit"]),
    "inner",
)
zeiss_matched_images.write.format("delta").mode("overwrite").save(
    str(match_root / "zeiss_matched_images_delta")
)
clsa_matched_images.write.format("delta").mode("overwrite").partitionBy(
    "visit"
).save(str(match_root / "clsa_matched_images_delta"))

run_summary = {
    "zeiss_source_images": int(len(zeiss_embeddings)),
    "zeiss_source_pipeline_embedded_images": int(len(zeiss_passed_images)),
    "zeiss_eligible_patients": int(len(zeiss_patients)),
    "zeiss_matched_patients": int(match_pairs["zeiss_patient_id"].nunique()),
    "clsa_matched_participants": int(match_pairs["clsa_participant_id"].nunique()),
    "clsa_embedding_source_mode": clsa_embedding_source_mode,
    "clsa_embedding_source": resolved_clsa_embeddings_path,
    "clsa_quality_source_mode": clsa_quality_source_mode,
    "clsa_quality_source": resolved_clsa_quality_path,
    "match_ratio": match_ratio,
    "age_caliper_years": age_caliper_years,
    "exact_sex_match": exact_sex_match,
    "mean_absolute_age_difference_years": float(
        match_pairs["absolute_age_difference_years"].mean()
    ),
    "zeiss_diagnosis_status": (
        "verified_column_filter_applied"
        if zeiss_diagnosis_column
        else "diagnosis_not_attached"
    ),
    "clsa_control_definition": (
        "visit-matched observed-negative released ocular screen plus no "
        "self-reported visual impairment"
    ),
}
(output_root / "AGE_GLAUCOMA_COHORT_SUMMARY.json").write_text(
    json.dumps(run_summary, indent=2), encoding="utf-8"
)
print(json.dumps(run_summary, indent=2))


%md
## Output map

```text
Age_Glaucoma/
├── 01_zeiss_source_cohort/
│   ├── zeiss_embedded_images.parquet
│   └── zeiss_patient_cohort.parquet
├── 03_clsa_controls/
│   ├── ocular_screen_delta/
│   └── eligible_images_delta/
├── 04_age_matching/
│   ├── age_match_pairs.parquet
│   ├── age_match_audit.parquet
│   ├── age_match_run_counts.csv
│   ├── age_match_balance.csv
│   ├── zeiss_matched_images_delta/
│   └── clsa_matched_images_delta/
└── AGE_GLAUCOMA_COHORT_SUMMARY.json
```
